In [1]:
from preprocessing import PDFTextExtractor
from indexing import PDFVectorStore
from model import  LLMQueryHandler
import os
from dotenv import load_dotenv
load_dotenv()

True

## PREPROCESSING THE CODE

In [2]:
pdf_path = "dataset/Serri_doc.pdf"  # Update with actual file path
extractor = PDFTextExtractor(pdf_path)
chunks = extractor.process_pdf()

## INDEXING 

In [3]:
pdf_store = PDFVectorStore()
pdf_store.create_vector_store(chunks)
pdf_store.save_vector_store()
new_vector_store = pdf_store.load_vector_store()

/home/sarveshharikant/anaconda3/envs/docetl/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## RETRIEVING THE CONTEXT 

In [4]:
query = "what is serri AI"
retrieved_document = new_vector_store.similarity_search(query,k=5)

In [5]:
page_contents = [doc.page_content for doc in retrieved_document]
print(page_contents)

['Title: Overview of Serri AI\n\nWhat is Serri AI? At Serri, We have developed infrastructure and an interface to train AI agents for any enterprise intelligence and execution use case. The AI agents work with your existing sales, marketing, and support systems. It’s an AI growth engine designed to supercharge B2C businesses. By automating end-to-end marketing and sales workflows through WhatsApp and other channels using advanced AI tools, Serri AI streamlines processes from lead generation and qualification to payment collection and post-sales engagement, creating a seamless customer journey. Mission: Serri AI\'s core mission is to empower small and medium-sized enterprises (SMEs) in emerging markets, including India, MENA, and SEA, by providing them with affordable, enterprise-grade growth tools. By eliminating the need for complex and costly CRMs or coding, Serri AI levels the playing field, making advanced growth strategies accessible to businesses of all sizes. Vision: Serri AI en

In [6]:

api_key = os.getenv("OPENAI_API_KEY")
print(api_key)

sk-djgKWsWicdUcI9E_XXzv8MzXuWEjsJdSG4Bfl0XRAhT3BlbkFJjUbv6yFI3qQT2uKz2nrY6foa5-KLORQq-_NYNv5AoA


## CHATBOT

In [7]:
api_key = os.getenv("OPENAI_API_KEY")
inference_model = LLMQueryHandler(api_key)

/home/sarveshharikant/EXIMIETAS/SARVESH/2025/serri/main_code/model.py:13: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  self.llm = ChatOpenAI(
/home/sarveshharikant/EXIMIETAS/SARVESH/2025/serri/main_code/model.py:19: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  self.llm_chain = LLMChain(prompt=self.prompt, llm=self.llm)


In [8]:
response = inference_model.generate_response(query,page_contents)
# print(response)

In [9]:
print(response.content)

Serri AI is an AI-powered platform designed to enhance the growth of B2C businesses by automating marketing and sales workflows through WhatsApp and other channels. It offers features like WhatsApp integration, AI-powered automation, and end-to-end workflows to streamline processes from lead generation to post-sales engagement, making sophisticated growth tools accessible and affordable for small and medium-sized enterprises (SMEs) in emerging markets.


## CREW AI AGENTS

In [10]:
from crewai import Agent, LLM, Task,Crew



# Advanced configuration with detailed parameters
llm = LLM(
    model="gpt-4-turbo",
    temperature=0,        # Higher for more creative outputs
    timeout=120,           # Seconds to wait for response
    max_tokens=4000,       # Maximum length of response
    top_p=0.9,            # Nucleus sampling parameter
    frequency_penalty=0.1, # Reduce repetition
    presence_penalty=0.1,  # Encourage topic diversity
    # response_format="text",  # For structured outputs
    seed=42  ,             # For reproducible results,
    api_key=os.getenv("OPENAI_API_KEY"),
)



## AGENT

In [11]:
output_validator = Agent(
    role = """
You are responsible for evaluating responses based on their relevance, accuracy, and usefulness in relation to the provided context.
Your assessment will help ensure high-quality outputs.
"""
,
goal = """
Your task is to analyze the generated response in comparison to the given context and provide a rating. 
Use the following scale to assess the quality of the output:
- "not-helpful" - The response is irrelevant, incorrect, or lacks meaningful information.
- "too-vague" - The response is somewhat related but lacks depth or clarity.
- "good" - The response is clear, informative, and well-aligned with the context.
"""
,
backstory = """
You are an experienced response evaluator with a deep understanding of assessing information quality.
Your expertise lies in identifying well-structured and meaningful outputs while filtering out vague or misleading responses.
"""
,
allow_delegation=False,
llm= llm,
verbose=False
)


## TASK

In [12]:
output_correction = Task(
    description=(
    """
Evaluation Task:
Assess the quality of the generated response based on the provided **input context**, **user prompt**, and **feedback**.

Information Provided:
- **Input Context: {input_context}  
- **User Prompt: {user_prompt}  
- **Generated Output: {output}  

Rating:
"not-helpful" - Incorrect, off-topic, or lacks meaningful content.
"too-vague" - Somewhat relevant but lacks depth or clarity.
"good" - Clear, informative, and contextually aligned.
    """
    ),
    expected_output=(
    """
    "Evaluation_Result": "RATING"
    """
    ),
    agent=output_validator
)




In [13]:
crew = Crew(
  agents=[output_validator],
  tasks=[output_correction],
  verbose=False,
  #memory=True
)

In [14]:
inputs = {

    "input_context": page_contents ,
    "user_prompt":query  , 
    "output": response,
}

In [15]:
result = crew.kickoff(inputs=inputs)


In [16]:
result.raw

'"Evaluation_Result": "good"'